In [ ]:
import keras
import numpy as np

path = keras.utils.get_file(
    'mobydick.txt', origin='https://www.gutenberg.org/files/2701/2701-0.txt')

text = open(path, encoding='utf-8').read().lower()
print('Corpus length: ', len(text))

1234609/1234609 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Corpus length:  1219043


In [ ]:
path = keras.utils.get_file(
    'folk_songs.txt', origin='https://www.gutenberg.org/files/1934/1934-0.txt')

text = open(path, encoding='utf-8').read().lower()
print('Corpus length: ', len(text))

52627/52627 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Corpus length:  50400


In [ ]:
maxlen = 60
step = 3

sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i + maxlen])
  next_chars.append(text[i + maxlen])

chars = sorted(list(set(text)))
char_indices = dict((char, chars.index(char)) for char in chars)

print('Number of sequences: ', len(sentences))
print('Unique characters: ', len(chars))

Number of sequences:  16780
Unique characters:  64


In [ ]:
x = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.bool)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool)

for i, sentence in enumerate(sentences):
  for t, char in enumerate(sentence):
    x[i, t, char_indices[char]] = 1
  y[i, char_indices[next_chars[i]]] = 1


In [ ]:
from keras import layers
model = keras.models.Sequential()
model.add(keras.Input(shape=(maxlen, len(chars))))

model.add(layers.LSTM(256, return_sequences=True))
model.add(layers.Dropout(0.2))
model.add(layers.LSTM(256))#Changed to 256
model.add(layers.Dropout(0.2))#Added dropout to prevent overfitting

model.add(layers.Dense(len(chars), activation='softmax'))

optimizer = keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer
)

In [ ]:
def sample(preds, temperature=1.0):
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temperature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1, preds, 1)
  return np.argmax(probas)

In [ ]:
import random
import sys

for epoch in range(1, 30):
  print('epoch ----------------------------------------------', epoch, '----------------------------------------------')
  model.fit(x, y, batch_size=128, epochs=1)

  start_index = random.randint(0, len(text) - maxlen - 1)
  generated_text = text[start_index: start_index + maxlen]
  print(' ---- Generating with seed: "'+generated_text+'"')
  for temperature in[0.2, 0.5, 1.0, 1.2]:
    print('----- temperature: ', temperature)
    sys.stdout.write(generated_text)

    for i in range(100):
      sampled = np.zeros((1, maxlen, len(chars)))
      for t, char in enumerate(generated_text):
        sampled[0, t, char_indices[char]] = 1.0

      preds = model.predict(sampled, verbose=0)[0]
      next_index = sample(preds, temperature)
      next_char = chars[next_index]

      generated_text += next_char
      generated_text = generated_text[1:]
      sys.stdout.write(next_char)
    print("\n")

epoch ---------------------------------------------- 1 ----------------------------------------------
132/132 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - loss: 3.2873
 ---- Generating with seed: "ibutions from states where we
have not met the solicitation "
----- temperature:  0.2
ibutions from states where we
have not met the solicitation              tu     i  e e    e t  t  n           e      n  s    ee    e t    e     e   l  e   e   t

----- temperature:  0.5
          e      n  s    ee    e t    e     e   l  e   e   tteiaen  t oe     re ei  tl be  nsenta   oeelc d io tntrt   in erneeeedtfio i ii
 rsrteeiae     an de

----- temperature:  1.0
oeelc d io tntrt   in erneeeedtfio i ii
 rsrteeiae     an de
r
hrm t ntrsnmtd
dcle;taiwwutsgohatooa‘too kbhfoaelekemeogd
smrleewen gls
mt
gfcih glun it

  septo

----- temperature:  1.2
too kbhfoaelekemeogd
smrleewen gls
mt
gfcih glun it

  septotge,l
setwfrlt. s
eytmfi.
y ugig d oryb iai h.rsh, li:eeh eg yssgg,uoswn wa w nfribneiato.rsn 
bs0 e

epoch 